### Before Spark

- In around 1995 Data Boom (Increase Volume, Velocity, and Variety)
- Vertical Scaling was consently being done to increase a single machine (Monolithic Approach)
  - Single point of failure
- Companies moved to distributed architectures --> Base for Hadoop 


**Hadoop**
- Hadoop is a framework used for storing and processing massive volumes of data in a distributed manner. 
- Consist of HDFS, MapReduce, and YARN 


**MapReduce**
- Processing Layer of Hadoop 
- Lets say 10 GB is distributed in HDFS into 8 files 
    - These 8 files are allocated to 8 Mappers (tasks)
    - Mapper will read the file and convert into key value pairs 
    - Key Value Pairs are stored in Local Storage in each machine (BOTTLENECK)
    - Once mappers are done processing, shuffling occurs 
    - Reducers then pull in key value pairs from Local Storage to make a finalized output 

### Spark

- To solve MapReduce Local Storage issue
- Spark is a open-source, in-memory, distributed data processing engine 
  - In Memory means up to 100x faster than MapReduce


**Architecture**
- Cluster Manager
  - Allocates resources to a Spark Application 
- Driver 
  - Runs the main Spark Application 
  - Generates execution plans and schedules tasks 
  - Coordinates executors and monitors execution
- Executors 
  - Executes tasks assigned by driver 
  - Processes partitions of data 
  - Stores intermediate results in memory 
  - Distributed JVM worker process 

- Flow 
  - User submits spark application 
  - Cluster Manager creates the driver node 
  - Driver node communicates with cluster manager to allocate executors 
  - Driver coordinates directly with executors for tasks 
  - Driver consolidates results 



### Jobs, Stages, and Tasks: 
**Jobs → Stages → Tasks**
- Spark breaks down distributed processing into three levels of execution 

- Job: A complete computation triggered by an action 
  - Each action can trigger one or more spark Jobs depending on the operation and execution plan 
- Stage: A group of tasks that can be executed without requiring shuffling between them 
  - Spark divides jobs into stages based on shuffle boundaries 
  - Narrow transformations executed within same stage 
  - Wide transformations introduce a shuffle, separating stages 
- Task: The smallest unit of execution, typically processing a single partition of data 
  - A task processes one partition within a stage 
  - If a stage has 100 partitions of data, spark typically creates 100 tasks for that stage 
  - Those tasks are distributed across available executor CPU slots 
  - If your cluster has 20 available task slots, approximately 20 tasks can execute concurrently 


### What is Databricks 

- Databricks is a unifed, cloud-based platform for data engineering, data analytics, and AI. 
- Founded by the creators of Spark, it provides a managed, scalable environment that connects data in one place 
- Breaks down silos so entire data team can use a single platform 

### Unity Catalog 

- Before UC ERA: 
  - Isolated workspaces each with own hive metastore and user management 
  - Lack of Data Lineage 
  

- After UC: 
  - Unified Metastore (Unity Metastore) and User Management across mulitple workspaces 
  - Three level namespace

### INSIDE: DRIVER and WORKERS 

**Driver Node:** 
- You submit PySpark Code 
- Inside the driver node (a JVM) your PySpark Code is converted to Java 
- SparkSession gets create 
- Plan gets created and submitted to internal DAG scheduler
  - Takes Logical Plan and converts it to stages 
  - Determines where shuffle boundaries will happen 
- DAG of transformations then submitted to Tasks Scheduler 
  - Recieves stages and then sends tasks to the executors 
  - Resubmits task on failures 
- Goes to Scheduler Backend which communicates with the Cluster Manager 
- Cluster Manager will provision executors 

**Worker Node:**
- A JVM consisting of Storage Memory (caching) and Execution Memory (for heavy computations like joins and aggregations)
- Executor Process 
  - Runs Multiple Tasks Concurrently

### Memory Management 

**Driver**
- Spark Driver Memory Management focuses on provisioning enough resources to orchestrate tasks, maintain metadata, and aggregate final results 
- Lets say you are asking for a driver with 2 GB of Memory, Spark allocates driver as follows: 
  - JVM Heap Memory: 
    - configured by spark.driver.memory
    - Gets the 2 GB of Memory 
    - Runs all execution code, maintains SparkContext, tracks data structures
  - Off Heap Memory Overhead: 
    - configured by spark.driver.memoryOverhead
    - allocates Off Heap memory as MAX(384MB or 0.1*Driver Memory)
    - It handles non-JVM processes, network buffers, and container overhead
- Driver Memory Management OOM Error 
  - Rarer then Executor OOM 
  - Lets say you run a df.collect() or df.take(), this will bring all partitions to the driver node 
    - If your Driver Heap Memory < the total of all partitions driver will be overloaded

___ 
**Executor**
- Lets say you request executor memory = 10 GB and 5 worker nodes 
  - JVM Heap 
    - 10 GB 
    - All of the executions 
    - 10 GBs gets further divided 
      - Reserved Memory 
        - Reserved by Spark at 300 MB for the spark Engine 
        - Non configurable
      - Unified Spark Memory
        - Out of remaining 9.7 GB, by default 60% (configurable via spark.memory.fraction) 
        - This is for core operations 
        - Split Further (50% each configurable via spark.memory.storageFraction): 
          - Storage Memory:
            - Used for caching 
          - Execution Memory: 
            - Used for execution of the tasks
            - Excution Memory takes precedents and can evict storage memory (based on least recently used caching)
      - User Memory
        - 40% of the left over 9.7 GB 
        - Stores UDFs, internal metadata, spark lineage transformations 
  - Off Heap Memory Overhead 
    - allocates Off Heap memory as MAX(384MB or 0.1*Driver Memory)
  - PySpark Memory
    - spark.driver.pyspark.memory
    - The memory consumed by the python processes evaluating your PySpark code or Pandas UDF 

### Wide and Narrow Transformations 
 - Main difference is whether data needs to move between partitions (shuffle) during processing 

**Narrow:** Each output partition depends on at most one input partition. Data stays within existing partition 
  - Efficient because: 
    - No network shuffle required 
    - Minimal data movement 
    - Multiple narrow transformations can be pipelined into same execution stage
    - Generally lower memory or disk overhead
    - EX: filter(), select(), withColumn(), map(), union()

**Wide:** An output partition depends on multiple input partitions
  - Expensive Because: 
    - Network I/O: transferring data between executors
    - Disk I/O: Writing and reading intermediate results 
    - Serialization: Converting data for transfer 
    - Memory Pressure: Buffering and sorting intermediate results 
    - Data Skew: Some partitions receive larger amounts of data 
    - EX: groupBy().agg(), distinct(), repartition(), orderBy(), join() (not always) 


### Lazy Evaluation: 

Spark does not immediately execute transformations. Instead, it builds an execution plan and waits until an action is called to process the data.
Allows spark to optimize the entire sequence of operations. 


- You submit code with transformations --> Unresolved Logical Plan 
- Optimized plan created using catalyst optimizer --> Resolved/Optimzed Plan 
- Creates Physical Plans --> Cost-Based Optimzed --> Tungsten Engine --> Physical Plan 

In [0]:
# Lazy Eval no data displayed 

data = [
    (1, "Dhruv", "Data Engineer"),
    (2, "John", "Software Engineer"),
    (3, "Jane", "Software Engineer"),
    (4, "Alice", "Data Analyst"),
    (5, "Bob", "ML Engineer"),
    (6, "Sam", "Product Manager"),
    (7, "Paul", "Data Analyst")
]

columns = ["id", "name", "role"]

df = spark.createDataFrame(data, columns)

In [0]:
df.display() # Action

In [0]:
df.explain()